# PlantDoc — Plant Disease Classifier v3

Rebuild of `plant-disease-v2` with the defects from that run fixed, the figures
redesigned, and a full evaluation / ablation suite added.

## What changed from v2

| # | Change | Why |
|---|---|---|
| 1 | `.cache()` moved **above** the augmentation map | In v2 augmentation ran once and was cached, so every epoch after the first saw identical images. Main cause of the train/val gap (train `1.0000`, val `0.956`). |
| 2 | Stratified **file-level** split | v2 used `take`/`skip` on a shuffled dataset — not reproducible, small boundary leak. |
| 3 | `EarlyStopping` added | v2 ran ~60 epochs past its best checkpoint. |
| 4 | Label smoothing 0.1 | Better calibrated confidence — the app shows this number to users. |
| 5 | `ReduceLROnPlateau` kept in **both** phases | Phase 1 halves on a 3-epoch plateau, phase 2 on a 5-epoch plateau — the schedule from v2, now paired with `EarlyStopping` so the run ends instead of idling at a dead LR. |
| 6 | Full backbone unfreeze in phase 2 | v2 unfroze only the top 100 of 155 layers. |
| 7 | `sqrt` class weighting (configurable) | v2's `balanced` weights hit 8.6x for a 152-image class. |
| 8 | Preprocessing **baked into the model** | Removes train/serve skew. Requires a backend change — see the final cell. |
| 9 | Test-time augmentation | Free accuracy at inference. |
| 10 | Configurable backbone | Inference runs server-side, so MobileNetV2 is not required. |
| 11 | Fabricated "augmented dataset" table removed | v2 reported +10,469 synthetic images that were never created. |

## Figure set

Every figure is numbered and saved to `/kaggle/working` for direct use in the
report.

| Figure | Content |
|---|---|
| Table 1 | Hyperparameters |
| Fig 1 | Class distribution by split |
| Fig 2 | Sample images and augmentation draws |
| Fig 3 | Training and validation loss / accuracy |
| Fig 4 | Validation accuracy vs training epochs |
| Fig 5 | Per-class precision, recall, F1 |
| Fig 6 | Confusion matrix |
| Fig 7 | Top confusions |
| Fig 8 | Error budget by species |
| Fig 9 | Confidence calibration and ECE |
| Fig 10 | Misclassified examples |
| Fig 11 | Convolutional feature maps |
| Fig 12 | Grad-CAM |
| Fig 13 | Accuracy / precision / recall / F1 vs classical ML baselines |
| Fig 14 | ROC curves, CNN vs baselines |
| Fig 15 | Ablation — dropout rate |
| Fig 16 | Ablation — batch size |
| Fig 17 | Ablation — augmented vs non-augmented |
| Fig 18 | Ablation — backbone comparison |

Figures 15-18 are **off by default** because each needs extra training runs.
Set `RUN_ABLATIONS = True` to produce them; the cell prints a time estimate.

**Removed from v2:** the 20-panel per-class ROC grid (18 panels showed only the
reference diagonal, because every curve hugged the corner), the three 0-to-1 bar
panels for per-class metrics (all bars looked identical), the five-bar summary
chart, and the 20-slice pie. ROC returns in Fig 14, where it separates models
that actually differ.

## 1. Configuration

`BACKBONE` is the main knob. Run `mobilenetv2` first for a clean comparison
against v2, then try `efficientnetv2b0`.

In [ ]:
import os

# --- data -----------------------------------------------------------------
DATA_DIR = ('/kaggle/input/datasets/nayanpokharel/plant-leaf-diseases-wa/'
            'Plant_leave_diseases_dataset_without_augmentation')
WORK_DIR = '/kaggle/working'

# --- backbone -------------------------------------------------------------
# 'mobilenetv2'      3.5M params, ImageNet top-1 71.8%  (v2 used this)
# 'efficientnetv2b0' 7.1M params, ImageNet top-1 78.7%  (recommended next step)
# 'efficientnetv2s'   21M params, ImageNet top-1 83.9%  (slowest to serve)
BACKBONE = 'mobilenetv2'

# --- data pipeline --------------------------------------------------------
IMG_SIZE = 224
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10
SHUFFLE_BUFFER = 4096
SEED = 42

# --- phase 1: train the head, backbone frozen -----------------------------
P1_LR = 1e-3
P1_EPOCHS = 20
P1_LR_FACTOR = 0.5     # halve the LR when val_loss plateaus
P1_LR_PATIENCE = 3
P1_MIN_LR = 1e-6
P1_EARLYSTOP_PATIENCE = 8

# --- phase 2: fine-tune ---------------------------------------------------
P2_LR = 5e-5           # starting LR; halved on plateau
P2_EPOCHS = 80
P2_LR_FACTOR = 0.5     # halve the LR when val_loss plateaus
P2_LR_PATIENCE = 5     # ...after 5 epochs without improvement
P2_MIN_LR = 1e-8
UNFREEZE_TOP = None    # None unfreezes the whole backbone; an int keeps the top N

# --- regularisation -------------------------------------------------------
DROPOUT_RATES = (0.4, 0.3, 0.2)
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
CLASS_WEIGHT_MODE = 'sqrt'   # 'balanced' | 'sqrt' | 'none'
EARLYSTOP_PATIENCE = 15

# --- inference ------------------------------------------------------------
USE_TTA = True

# --- comparison figures ---------------------------------------------------
# Classical ML baselines for Fig 13 / Fig 14. Cheap (~5 min).
RUN_BASELINES = True
BASELINE_IMG_SIZE = 32       # images are flattened to 32x32 greyscale
BASELINE_TRAIN_N = 8000      # subsample: SVM/KNN scale badly past this

# Ablation studies for Fig 15-18. Each entry is an extra training run.
# Roughly 60-90 min on a T4 with the defaults below. Off by default.
RUN_ABLATIONS = False
ABLATION_EPOCHS = 8
ABLATION_TRAIN_N = 6000
ABLATION_DROPOUTS = [0.0, 0.2, 0.4, 0.6, 0.8]
ABLATION_BATCH_SIZES = [16, 32, 64, 128]
ABLATION_BACKBONES = ['mobilenetv2', 'efficientnetv2b0', 'resnet50', 'vgg16']

MODEL_VERSION = f'{BACKBONE}-20c-v3'
BEST_MODEL_PATH = os.path.join(WORK_DIR, 'plant_best_model.keras')
os.makedirs(WORK_DIR, exist_ok=True)

print(f'backbone       : {BACKBONE}')
print(f'model version  : {MODEL_VERSION}')
print(f'class weights  : {CLASS_WEIGHT_MODE}')
print(f'label smoothing: {LABEL_SMOOTHING}')
print(f'baselines      : {RUN_BASELINES}')
print(f'ablations      : {RUN_ABLATIONS}')

## 2. Imports, seeds, environment

In [ ]:
import itertools
import json
import math
import random
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_curve, auc)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

keras.mixed_precision.set_global_policy('mixed_float16')
AUTOTUNE = tf.data.AUTOTUNE

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

C_BLUE, C_GREEN, C_RED, C_PURPLE, C_SAND = (
    '#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974')
C_ORANGE, C_TEAL = '#DD8452', '#64B5CD'

print('TensorFlow      :', tf.__version__)
print('Keras           :', keras.__version__)
print('Mixed precision :', keras.mixed_precision.global_policy().compute_dtype)
print('GPUs            :', tf.config.list_physical_devices('GPU'))

## 3. Index the dataset

Class order must stay byte-identical to the deployed `Resources/class_names.json`,
because the label-to-index mapping is baked into the trained weights. Sorted
directory names is what `image_dataset_from_directory` used in v2, so sorting
here reproduces it. The assert is the guard.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

data_root = Path(DATA_DIR)
if not data_root.exists():
    candidates = [p for p in Path('/kaggle/input').rglob('*')
                  if p.is_dir() and any(c.name.endswith('___healthy')
                                        for c in p.iterdir() if c.is_dir())]
    if not candidates:
        raise FileNotFoundError(f'Could not locate the dataset under {DATA_DIR}')
    data_root = candidates[0]
    print(f'DATA_DIR not found; using discovered path: {data_root}')

CLASS_NAMES = sorted(d.name for d in data_root.iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)

all_paths, all_labels = [], []
for idx, cname in enumerate(CLASS_NAMES):
    for p in sorted((data_root / cname).iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            all_paths.append(str(p))
            all_labels.append(idx)

all_paths = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int32)
counts = np.bincount(all_labels, minlength=NUM_CLASSES)
print(f'Classes: {NUM_CLASSES} | Images: {len(all_paths):,}')

DEPLOYED_ORDER = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust',
    'Apple___healthy', 'Background_without_leaves', 'Potato___Early_blight',
    'Potato___Late_blight', 'Potato___healthy', 'Strawberry___Leaf_scorch',
    'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight',
    'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy',
]
assert CLASS_NAMES == DEPLOYED_ORDER, (
    'Class order differs from the deployed model. Every prediction the backend '
    'makes would be mislabelled. Resolve before training.\n'
    f'got:      {CLASS_NAMES}\nexpected: {DEPLOYED_ORDER}')
print('Class order matches the deployed class_names.json.')

SHORT_NAMES = [c.replace('___', ' — ').replace('_', ' ') for c in CLASS_NAMES]

## 4. Stratified split

v2 used `take`/`skip` on a shuffled `tf.data` pipeline. Because train, val and
test were each `.cache()`d on separate passes and the shuffle buffer reshuffles
per iteration, membership near the boundaries was unstable and the split could
not be reproduced. Splitting the file list makes the partition exact,
stratified and reproducible. The assertions are the proof.

In [ ]:
trainval_p, test_p, trainval_l, test_l = train_test_split(
    all_paths, all_labels, test_size=TEST_SPLIT,
    stratify=all_labels, random_state=SEED)

val_fraction = VAL_SPLIT / (1.0 - TEST_SPLIT)
train_p, val_p, train_l, val_l = train_test_split(
    trainval_p, trainval_l, test_size=val_fraction,
    stratify=trainval_l, random_state=SEED)

train_set, val_set, test_set = set(train_p), set(val_p), set(test_p)
assert not (train_set & val_set), 'train/val overlap'
assert not (train_set & test_set), 'train/test overlap'
assert not (val_set & test_set), 'val/test overlap'
assert len(train_set) + len(val_set) + len(test_set) == len(all_paths)
print('Split verified: no file appears in more than one split.\n')

print(f'{"split":<8}{"images":>9}{"share":>9}')
for name, arr in [('train', train_p), ('val', val_p), ('test', test_p)]:
    print(f'{name:<8}{len(arr):>9,}{len(arr) / len(all_paths):>8.1%}')

STEPS_PER_EPOCH = math.ceil(len(train_p) / BATCH_SIZE)
print(f'\nsteps/epoch: {STEPS_PER_EPOCH}')

train_counts = np.bincount(train_l, minlength=NUM_CLASSES).astype(np.float64)
if CLASS_WEIGHT_MODE == 'balanced':
    w = len(train_l) / (NUM_CLASSES * train_counts)
elif CLASS_WEIGHT_MODE == 'sqrt':
    w = np.sqrt(len(train_l) / (NUM_CLASSES * train_counts))
elif CLASS_WEIGHT_MODE == 'none':
    w = np.ones(NUM_CLASSES)
else:
    raise ValueError(f'unknown CLASS_WEIGHT_MODE: {CLASS_WEIGHT_MODE}')

CLASS_WEIGHTS = (w / w.mean()).astype(np.float32)
CW_TENSOR = tf.constant(CLASS_WEIGHTS, dtype=tf.float32)

print(f'\nclass weights ({CLASS_WEIGHT_MODE}), '
      f'range {CLASS_WEIGHTS.min():.2f}-{CLASS_WEIGHTS.max():.2f}')

## 5. Figure 1 — class distribution

v2 plotted a real "Original" series beside a fabricated "Augmented" series
(`int(count * 1.4)`; those images never existed), plus a 20-slice pie with
colliding labels. Replaced with one sorted horizontal bar chart of real counts,
coloured by split so the stratification is visible.

In [ ]:
val_counts = np.bincount(val_l, minlength=NUM_CLASSES)
test_counts = np.bincount(test_l, minlength=NUM_CLASSES)

order = np.argsort(counts)
y = np.arange(NUM_CLASSES)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(y, train_counts[order], color=C_BLUE, label='train')
ax.barh(y, val_counts[order], left=train_counts[order], color=C_GREEN,
        label='val')
ax.barh(y, test_counts[order], left=train_counts[order] + val_counts[order],
        color=C_SAND, label='test')
for i, idx in enumerate(order):
    ax.text(counts[idx] + counts.max() * 0.01, i, f'{counts[idx]:,}',
            va='center', fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels([SHORT_NAMES[i] for i in order], fontsize=9)
ax.set_xlabel('Images')
ax.set_xlim(0, counts.max() * 1.12)
ax.set_title(f'Fig 1 — Class distribution ({len(all_paths):,} images, '
             f'{NUM_CLASSES} classes)', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig01_class_distribution.png', bbox_inches='tight')
plt.show()

print(f'Imbalance ratio: {counts.max() / counts.min():.1f}x  '
      f'({CLASS_NAMES[counts.argmax()]} vs {CLASS_NAMES[counts.argmin()]})')

## 6. Input pipeline — the v2 bug fixed

Ordering is the whole point:

```
load -> cache -> shuffle -> batch -> AUGMENT -> prefetch
                                     ^
                                     runs fresh every epoch
```

v2 had `map(augment) -> cache()`, which froze one random draw per image for the
whole run. The tell in the v2 logs was epoch 1 at 259s and epoch 2 at 14s.
Expect roughly uniform epoch times here.

Images cache as **uint8** (~3 GB) rather than float32 (~12 GB), in the raw
0-255 range, because preprocessing now lives inside the model.

In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal_and_vertical'),
    keras.layers.RandomRotation(0.2),
    keras.layers.RandomZoom(0.15),
    keras.layers.RandomTranslation(0.1, 0.1),
    keras.layers.RandomContrast(0.2),
    keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
], name='augmentation')


def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3,
                             expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bilinear')
    img = tf.cast(tf.round(tf.clip_by_value(img, 0.0, 255.0)), tf.uint8)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label


def augment_and_finalize(img, label):
    img = tf.cast(img, tf.float32)
    img = augmentation(img, training=True)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, tf.one_hot(label, NUM_CLASSES), tf.gather(CW_TENSOR, label)


def plain_and_finalize(img, label):
    return (tf.cast(img, tf.float32), tf.one_hot(label, NUM_CLASSES),
            tf.gather(CW_TENSOR, label))


def finalize_eval(img, label):
    return tf.cast(img, tf.float32), tf.one_hot(label, NUM_CLASSES)


def make_train_ds(paths, labels, augment=True, batch_size=None):
    bs = batch_size or BATCH_SIZE
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.shuffle(min(SHUFFLE_BUFFER, len(paths)), seed=SEED,
                    reshuffle_each_iteration=True)
    ds = ds.batch(bs)
    ds = ds.map(augment_and_finalize if augment else plain_and_finalize,
                num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def make_eval_ds(paths, labels, batch_size=None):
    # No shuffle: prediction order must line up with `labels` later.
    bs = batch_size or BATCH_SIZE
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()
    ds = ds.batch(bs)
    ds = ds.map(finalize_eval, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_train_ds(train_p, train_l)
val_ds = make_eval_ds(val_p, val_l)
test_ds = make_eval_ds(test_p, test_l)

xb, yb, wb = next(iter(train_ds))
print(f'train batch : {xb.shape} {xb.dtype}  range [{tf.reduce_min(xb):.1f}, '
      f'{tf.reduce_max(xb):.1f}]')
print(f'labels      : {yb.shape} (one-hot)')
print(f'sample wts  : {wb.shape}  range [{tf.reduce_min(wb):.2f}, '
      f'{tf.reduce_max(wb):.2f}]')

## 7. Figure 2 — samples and augmentation

Top row: raw training images. Bottom row: eight augmentation draws of a single
image. If those eight tiles are ever identical, the v2 caching bug has
returned.

In [ ]:
fig = plt.figure(figsize=(18, 5.6))
gs = fig.add_gridspec(2, 8, hspace=0.18)

sample_ds = make_eval_ds(train_p[:8], train_l[:8])
imgs, onehots = next(iter(sample_ds.unbatch().batch(8)))
for k in range(8):
    ax = fig.add_subplot(gs[0, k])
    ax.imshow(imgs[k].numpy().astype(np.uint8))
    ax.set_title(SHORT_NAMES[int(tf.argmax(onehots[k]))], fontsize=7)
    ax.axis('off')
    if k == 0:
        ax.text(-0.12, 0.5, '(a) samples', transform=ax.transAxes,
                rotation=90, va='center', ha='center', fontsize=10,
                fontweight='bold')

probe_ds = tf.data.Dataset.from_tensor_slices((train_p[:1], train_l[:1]))
probe_img, probe_lbl = next(iter(probe_ds.map(load_image).batch(1)))
for k in range(8):
    ax = fig.add_subplot(gs[1, k])
    aug = augmentation(tf.cast(probe_img, tf.float32), training=True)
    ax.imshow(tf.clip_by_value(aug, 0, 255)[0].numpy().astype(np.uint8))
    ax.axis('off')
    if k == 0:
        ax.text(-0.12, 0.5, '(b) augmented', transform=ax.transAxes,
                rotation=90, va='center', ha='center', fontsize=10,
                fontweight='bold')

plt.suptitle('Fig 2 — Training samples and augmentation draws',
             fontsize=13, fontweight='bold')
plt.savefig(f'{WORK_DIR}/fig02_samples_augmentation.png', bbox_inches='tight')
plt.show()

## 8. Model

Two structural differences from v2.

**Preprocessing is inside the model.** v2 saved a model expecting inputs already
mapped to [-1, 1], leaving `preprocess_input` as a convention held only by the
serving code — a mismatch there produces confident wrong answers with no error.
Here the model takes raw 0-255 pixels and rescales internally, so the contract
travels with the weights. **This requires a backend change** — see the final
cell.

**`backbone(x, training=False)` is deliberate.** It bakes inference-mode
BatchNorm into the graph, so the backbone's moving statistics are never updated
during fine-tuning. Recommended practice, and what v2 did accidentally by
reusing its `trainable` flag as the `training` argument.

In [ ]:
BACKBONES = {
    'mobilenetv2': dict(
        builder=lambda: keras.applications.MobileNetV2(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet'),
        preprocess=lambda: keras.layers.Rescaling(1 / 127.5, offset=-1,
                                                  name='preprocess'),
    ),
    'efficientnetv2b0': dict(
        builder=lambda: keras.applications.EfficientNetV2B0(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet', include_preprocessing=True),
        preprocess=lambda: keras.layers.Identity(name='preprocess'),
    ),
    'efficientnetv2s': dict(
        builder=lambda: keras.applications.EfficientNetV2S(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet', include_preprocessing=True),
        preprocess=lambda: keras.layers.Identity(name='preprocess'),
    ),
    # Used only by the Fig 18 backbone ablation.
    'resnet50': dict(
        builder=lambda: keras.applications.ResNet50(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet'),
        preprocess=lambda: keras.layers.Lambda(
            keras.applications.resnet50.preprocess_input, name='preprocess'),
    ),
    'vgg16': dict(
        builder=lambda: keras.applications.VGG16(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet'),
        preprocess=lambda: keras.layers.Lambda(
            keras.applications.vgg16.preprocess_input, name='preprocess'),
    ),
    'inceptionv3': dict(
        builder=lambda: keras.applications.InceptionV3(
            input_shape=IMG_SHAPE, include_top=False, pooling='avg',
            weights='imagenet'),
        preprocess=lambda: keras.layers.Lambda(
            keras.applications.inception_v3.preprocess_input,
            name='preprocess'),
    ),
}


def build_model(num_classes, trainable_backbone=False, dropouts=None,
                backbone_name=None):
    name = backbone_name or BACKBONE
    rates = dropouts if dropouts is not None else DROPOUT_RATES
    spec = BACKBONES[name]
    backbone = spec['builder']()
    backbone.trainable = trainable_backbone

    inputs = keras.Input(shape=IMG_SHAPE, name='raw_rgb_0_255')
    x = spec['preprocess']()(inputs)
    x = backbone(x, training=False)
    x = keras.layers.Dropout(rates[0])(x)
    x = keras.layers.Dense(512, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(rates[1])(x)
    x = keras.layers.Dense(256, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(rates[2])(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax',
                                 dtype='float32', name='predictions')(x)
    return keras.Model(inputs, outputs, name=f'plant_{name}')


def get_backbone(m):
    return next(l for l in m.layers if isinstance(l, keras.Model))


def param_report(m, tag):
    trainable = int(sum(np.prod(v.shape) for v in m.trainable_variables))
    total = m.count_params()
    print(f'{tag:<24} trainable {trainable:>12,} / {total:,} '
          f'({100 * trainable / total:.1f}%)')
    return trainable, total


loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

model = build_model(NUM_CLASSES, trainable_backbone=False)
P1_TRAINABLE, TOTAL_PARAMS = param_report(model, 'phase 1 (head only)')
model.summary()

## 9. Table 1 — hyperparameters

Rendered as an image as well as printed, so it drops straight into the report.

In [ ]:
HYPERPARAMS = [
    ('Backbone', BACKBONE),
    ('Input resolution', f'{IMG_SIZE} x {IMG_SIZE} x 3'),
    ('Total parameters', f'{TOTAL_PARAMS:,}'),
    ('Phase 1 epochs (head only)', str(P1_EPOCHS)),
    ('Phase 1 learning rate', f'{P1_LR:g}'),
    ('Phase 2 epochs (fine-tune)', str(P2_EPOCHS)),
    ('Phase 2 starting learning rate', f'{P2_LR:g}'),
    ('LR schedule', f'ReduceLROnPlateau x{P2_LR_FACTOR:g} '
                    f'on val_loss'),
    ('LR patience (phase 1 / phase 2)',
     f'{P1_LR_PATIENCE} / {P2_LR_PATIENCE} epochs'),
    ('Minimum learning rate', f'{P2_MIN_LR:g}'),
    ('Optimiser', 'AdamW (weight decay 1e-4, clipnorm 1.0)'),
    ('Loss', f'Categorical cross-entropy, smoothing {LABEL_SMOOTHING}'),
    ('Mini-batch size', str(BATCH_SIZE)),
    ('Dropout rates', ' / '.join(str(d) for d in DROPOUT_RATES)),
    ('Class weighting', CLASS_WEIGHT_MODE),
    ('Early stopping patience', str(EARLYSTOP_PATIENCE)),
    ('Training set size', f'{len(train_p):,}'),
    ('Validation set size', f'{len(val_p):,}'),
    ('Test set size', f'{len(test_p):,}'),
    ('Random seed', str(SEED)),
]

print('=' * 64)
print('  Table 1 — Hyperparameters of the proposed model')
print('=' * 64)
for k, v in HYPERPARAMS:
    print(f'  {k:<32}{v:>28}')
print('=' * 64)

fig, ax = plt.subplots(figsize=(9, 0.42 * len(HYPERPARAMS) + 1))
ax.axis('off')
tbl = ax.table(cellText=[[k, v] for k, v in HYPERPARAMS],
               colLabels=['Parameter', 'Value'],
               cellLoc='left', colLoc='left', loc='center',
               colWidths=[0.55, 0.45])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.45)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#DDDDDD')
    if r == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#EEF2E7')
    elif r % 2 == 0:
        cell.set_facecolor('#FAFAFA')
ax.set_title('Table 1 — Hyperparameters of the proposed model',
             fontweight='bold', pad=18)
plt.savefig(f'{WORK_DIR}/tbl01_hyperparameters.png', bbox_inches='tight')
plt.show()

## 10. Phase 1 — head only

Short, fixed LR, backbone frozen. Enough to give phase 2 a sane starting point
rather than a randomly initialised head that would wreck pretrained features on
the first gradient step.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=P1_LR),
    loss=loss_fn,
    metrics=[keras.metrics.CategoricalAccuracy(name='acc')],
)

p1_callbacks = [
    keras.callbacks.ModelCheckpoint(
        f'{WORK_DIR}/p1_best.keras', monitor='val_acc',
        save_best_only=True, mode='max', verbose=0),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=P1_LR_FACTOR, patience=P1_LR_PATIENCE,
        min_lr=P1_MIN_LR, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_acc', mode='max', patience=P1_EARLYSTOP_PATIENCE,
        restore_best_weights=True, verbose=1),
    keras.callbacks.CSVLogger(f'{WORK_DIR}/p1_log.csv'),
]

print(f'Phase 1 — head only, up to {P1_EPOCHS} epochs\n')
t0 = time.time()
history_p1 = model.fit(train_ds, validation_data=val_ds,
                       epochs=P1_EPOCHS, callbacks=p1_callbacks, verbose=1)
print(f'\nPhase 1 best val_acc: {max(history_p1.history["val_acc"]) * 100:.2f}% '
      f'({time.time() - t0:.0f}s)')

## 11. Phase 2 — fine-tune

Full unfreeze with AdamW, starting at `P2_LR` and **halving the learning rate
whenever `val_loss` goes 5 epochs without improving** — the schedule from v2,
which is what an adaptive plateau schedule should look like.

The learning rate is a plain float here, not a schedule object: `ReduceLROnPlateau`
works by mutating `optimizer.learning_rate`, which a schedule object would
override.

The one thing v2 lacked is `EarlyStopping`. Without it, v2 kept training ~60
epochs past its best checkpoint at a learning rate of 4.88e-8, changing nothing.
Plateau reduction decides *how fast* to learn; early stopping decides *when to
stop*. You want both.

In [ ]:
backbone = get_backbone(model)
backbone.trainable = True
if UNFREEZE_TOP is not None:
    for layer in backbone.layers[:-UNFREEZE_TOP]:
        layer.trainable = False
    print(f'Unfroze the top {UNFREEZE_TOP} of {len(backbone.layers)} layers')
else:
    print(f'Unfroze all {len(backbone.layers)} backbone layers')

param_report(model, 'phase 2 (fine-tune)')

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=P2_LR, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
    loss=loss_fn,
    metrics=[keras.metrics.CategoricalAccuracy(name='acc')],
)

p2_callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor='val_acc',
        save_best_only=True, mode='max', verbose=0),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=P2_LR_FACTOR, patience=P2_LR_PATIENCE,
        min_lr=P2_MIN_LR, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_acc', mode='max', patience=EARLYSTOP_PATIENCE,
        restore_best_weights=True, verbose=1),
    keras.callbacks.CSVLogger(f'{WORK_DIR}/p2_log.csv'),
]

print(f'\nPhase 2 — fine-tune, up to {P2_EPOCHS} epochs '
      f'(start LR {P2_LR}, halve after {P2_LR_PATIENCE} flat epochs)\n')
t0 = time.time()
history_p2 = model.fit(train_ds, validation_data=val_ds,
                       epochs=P2_EPOCHS, callbacks=p2_callbacks, verbose=1)
print(f'\nPhase 2 best val_acc: {max(history_p2.history["val_acc"]) * 100:.2f}% '
      f'({time.time() - t0:.0f}s)')

## 12. Figure 3 — training and validation curves

Four fixes over v2: log-scale loss (v2's linear axis compressed phase 2 into a
flat line), a zoomed accuracy panel (v2's 0-1 axis made train accuracy a
featureless line at 1.0), a shaded generalisation gap so overfitting is legible
rather than inferred, and a **learning-rate panel** showing exactly which epochs
triggered a halving.

v2's dead `has_pr` branch — testing for `'precision' in hist` when only `acc`
was compiled, so the third panel never drew — is gone.

In [ ]:
h1, h2 = history_p1.history, history_p2.history
p1_len = len(h1['acc'])
ep = np.arange(1, p1_len + len(h2['acc']) + 1)

tr_loss = np.array(h1['loss'] + h2['loss'])
va_loss = np.array(h1['val_loss'] + h2['val_loss'])
tr_acc = np.array(h1['acc'] + h2['acc'])
va_acc = np.array(h1['val_acc'] + h2['val_acc'])

# ReduceLROnPlateau logs 'learning_rate' each epoch; absent if it never ran.
lr_hist = None
if 'learning_rate' in h1 and 'learning_rate' in h2:
    lr_hist = np.array(list(h1['learning_rate']) + list(h2['learning_rate']),
                       dtype=float)

n_panels = 3 if lr_hist is not None else 2
fig, axes = plt.subplots(1, n_panels, figsize=(7.5 * n_panels, 5.2))

ax = axes[0]
ax.plot(ep, tr_loss, color=C_BLUE, lw=2, label='train')
ax.plot(ep, va_loss, color=C_RED, lw=2, label='validation')
ax.fill_between(ep, tr_loss, va_loss, where=va_loss >= tr_loss,
                color=C_RED, alpha=0.10, label='generalisation gap')
ax.axvline(p1_len + 0.5, color='#888', ls='--', lw=1.2)
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('(a) Loss', fontweight='bold')
ax.legend()

ax = axes[1]
ax.plot(ep, tr_acc, color=C_BLUE, lw=2, label='train')
ax.plot(ep, va_acc, color=C_RED, lw=2, label='validation')
ax.fill_between(ep, va_acc, tr_acc, where=tr_acc >= va_acc,
                color=C_RED, alpha=0.10)
ax.axvline(p1_len + 0.5, color='#888', ls='--', lw=1.2)
best_ep = int(np.argmax(va_acc)) + 1
ax.scatter([best_ep], [va_acc.max()], color=C_GREEN, s=80, zorder=5,
           label=f'best val {va_acc.max():.4f} @ epoch {best_ep}')
acc_lo = min(0.80, float(min(va_acc.min(), tr_acc.min())) - 0.05)
ax.set_ylim(acc_lo, 1.005)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title(f'(b) Accuracy (zoomed {acc_lo:.2f}-1.00)', fontweight='bold')
ax.legend(loc='lower right')

if lr_hist is not None:
    ax = axes[2]
    ax.step(ep, lr_hist, where='post', color=C_PURPLE, lw=2)
    # i == p1_len is the phase 1 -> phase 2 LR change, not a plateau reduction.
    drops = [int(i) for i in range(1, len(lr_hist))
             if i != p1_len and lr_hist[i] < lr_hist[i - 1] * 0.99]
    for d in drops:
        ax.axvline(d + 1, color=C_RED, ls=':', lw=1.2, alpha=0.8)
    ax.axvline(p1_len + 0.5, color='#888', ls='--', lw=1.2)
    ax.set_yscale('log')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning rate (log scale)')
    ax.set_title(f'(c) Learning rate — {len(drops)} plateau reductions '
                 f'(red dotted)', fontweight='bold')

plt.suptitle('Fig 3 — Training history (grey dashed = fine-tune start)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig03_training_history.png', bbox_inches='tight')
plt.show()

if lr_hist is not None:
    print(f'Learning rate: {lr_hist[0]:.2g} -> {lr_hist[-1]:.2g} '
          f'in {len(drops)} reductions'
          + (f' (epochs {", ".join(str(d + 1) for d in drops)})'
             if drops else ''))

final_gap = tr_acc[-1] - va_acc[-1]
print(f'Final train acc {tr_acc[-1]:.4f} | val acc {va_acc[-1]:.4f} | '
      f'gap {final_gap:.4f}')
print('v2 finished with a gap of 0.0441 (train 1.0000, val 0.9559).')
if final_gap > 0.03:
    print('Gap still above 0.03: consider stronger augmentation or more dropout.')

## 13. Figure 4 — validation accuracy vs training epochs

Read directly off the history, so it costs nothing extra. Shows how much of the
final accuracy each epoch budget would have bought, which is the question the
"epochs comparison" figure in the literature is really answering.

In [ ]:
marks = [m for m in [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100]
         if m <= len(va_acc)]
if marks[-1] != len(va_acc):
    marks.append(len(va_acc))
best_by = [float(va_acc[:m].max()) for m in marks]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar([str(m) for m in marks], best_by, color=C_TEAL, alpha=0.9,
              edgecolor='white')
for b, v in zip(bars, best_by):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.004, f'{v:.4f}',
            ha='center', fontsize=8, rotation=90)

lo = min(best_by)
ax.set_ylim(max(0, lo - 0.05), 1.03)
ax.axhline(va_acc.max(), color=C_RED, ls='--', lw=1.2,
           label=f'best achieved {va_acc.max():.4f}')
ax.set_xlabel('Training epochs')
ax.set_ylabel('Best validation accuracy so far')
ax.set_title('Fig 4 — Validation accuracy vs training epochs',
             fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig04_epochs_comparison.png', bbox_inches='tight')
plt.show()

reach = next((m for m, v in zip(marks, best_by) if v >= va_acc.max() - 0.005),
             marks[-1])
print(f'Within 0.5 pts of the best result by epoch {reach} '
      f'of {len(va_acc)} ({reach / len(va_acc):.0%} of the budget).')

## 14. Evaluate, with and without TTA

TTA averages the softmax over the image, its horizontal flip and its vertical
flip. Costs 3x inference, needs no retraining. The comparison tells you whether
to enable it in the backend.

In [ ]:
model = keras.models.load_model(BEST_MODEL_PATH)
print(f'Loaded best checkpoint: {BEST_MODEL_PATH}\n')

base_loss, base_acc = model.evaluate(test_ds, verbose=1)
print(f'\nPlain test accuracy: {base_acc * 100:.2f}%  (loss {base_loss:.4f})')


def predict_proba(ds, tta=False):
    out = []
    for images, _ in ds:
        p = model(images, training=False)
        if tta:
            p = p + model(tf.image.flip_left_right(images), training=False)
            p = p + model(tf.image.flip_up_down(images), training=False)
            p = p / 3.0
        out.append(np.asarray(p, dtype=np.float32))
    return np.concatenate(out, axis=0)


y_true = test_l.copy()
proba_plain = predict_proba(test_ds, tta=False)
acc_plain = float((proba_plain.argmax(1) == y_true).mean())

if USE_TTA:
    proba_tta = predict_proba(test_ds, tta=True)
    acc_tta = float((proba_tta.argmax(1) == y_true).mean())
    print(f'\nplain : {acc_plain * 100:.2f}%')
    print(f'TTA   : {acc_tta * 100:.2f}%   '
          f'(delta {(acc_tta - acc_plain) * 100:+.2f} pts)')
    use_tta_final = acc_tta >= acc_plain
else:
    proba_tta, acc_tta, use_tta_final = None, None, False

y_prob = proba_tta if use_tta_final else proba_plain
y_pred = y_prob.argmax(1)
y_conf = y_prob.max(1)
correct = (y_pred == y_true)
overall_acc = float(correct.mean())

print(f'\nReporting with TTA: {use_tta_final}')
print(f'Test samples : {len(y_true):,}')
print(f'Accuracy     : {overall_acc * 100:.2f}%')
print(f'Errors       : {int((~correct).sum())}')

## 15. Figure 5 — per-class precision, recall, F1

v2 drew three panels of horizontal bars on a 0-1 axis. Every bar looked the same
length because every score sat between 0.81 and 1.00, and the 20 y-labels
overlapped. One dot plot, axis clipped to where the data lives, worst-first.

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(NUM_CLASSES), zero_division=0)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                            digits=4, zero_division=0))

order = np.argsort(f1)
y = np.arange(NUM_CLASSES)
lo = max(0.0, min(precision.min(), recall.min(), f1.min()) - 0.03)

fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y, lo, 1.0, color='#EAEAEA', lw=1.2, zorder=1)
for vals, lbl, c in [(precision, 'Precision', C_BLUE),
                     (recall, 'Recall', C_GREEN),
                     (f1, 'F1', C_RED)]:
    ax.scatter(vals[order], y, s=55, color=c, label=lbl, zorder=3)

ax.set_yticks(y)
ax.set_yticklabels([f'{SHORT_NAMES[i]}  (n={support[i]})' for i in order],
                   fontsize=8.5)
ax.set_xlim(lo, 1.005)
ax.invert_yaxis()   # worst F1 at the top, matching the title
ax.set_xlabel('Score')
ax.set_title(f'Fig 5 — Per-class precision / recall / F1, worst first '
             f'(axis starts at {lo:.2f})', fontweight='bold')
ax.legend(loc='lower left')
ax.grid(axis='y', alpha=0)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig05_per_class_metrics.png', bbox_inches='tight')
plt.show()

print('Weakest five by F1:')
for i in order[:5]:
    print(f'  {CLASS_NAMES[i]:<50} P={precision[i]:.3f} R={recall[i]:.3f} '
          f'F1={f1[i]:.3f}  n={support[i]}')

## 16. Figure 6 — confusion matrix

v2 annotated all 400 cells with `count` and `(fraction)`. About 380 read
`0 (0.00)`, burying the ~20 that carried information, and the diagonal text was
white-on-dark and unreadable. Only non-zero cells are annotated here.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.045, pad=0.03, label='Row-normalised')

ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(SHORT_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(SHORT_NAMES, fontsize=8)

for i, j in itertools.product(range(NUM_CLASSES), range(NUM_CLASSES)):
    if cm[i, j] == 0:
        continue
    ax.text(j, i, cm[i, j], ha='center', va='center',
            fontsize=8, fontweight='bold' if i == j else 'normal',
            color='white' if cm_norm[i, j] > 0.55
            else (C_RED if i != j else '#333'))

ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title(f'Fig 6 — Confusion matrix, counts, non-zero cells only '
             f'({overall_acc * 100:.2f}% accuracy)', fontweight='bold')
ax.grid(alpha=0)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig06_confusion_matrix.png', bbox_inches='tight')
plt.show()

## 17. Figure 7 — top confusions

Which pairs does the model actually mix up, and how often.

In [ ]:
pairs = [(i, j, int(cm[i, j]))
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
         if i != j and cm[i, j] > 0]
pairs.sort(key=lambda t: -t[2])
top = pairs[:14]

if top:
    labels = [f'{CLASS_NAMES[i].split("___")[-1].replace("_", " ")}  ->  '
              f'{CLASS_NAMES[j].split("___")[-1].replace("_", " ")}'
              for i, j, _ in top]
    vals = [c for _, _, c in top]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(range(len(top))[::-1], vals, color=C_RED, alpha=0.85)
    ax.set_yticks(range(len(top))[::-1])
    ax.set_yticklabels(labels, fontsize=9)
    for b, v in zip(bars, vals):
        ax.text(v + max(vals) * 0.015, b.get_y() + b.get_height() / 2,
                str(v), va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Misclassified images')
    ax.set_xlim(0, max(vals) * 1.15)
    ax.set_title(f'Fig 7 — Top confusions ({int((~correct).sum())} errors total)',
                 fontweight='bold')
    ax.grid(axis='y', alpha=0)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig07_top_confusions.png', bbox_inches='tight')
    plt.show()

healthy_idx = {i for i, c in enumerate(CLASS_NAMES) if c.endswith('___healthy')}
missed = int(sum(cm[i, j] for i in range(NUM_CLASSES) for j in healthy_idx
                 if i not in healthy_idx and i != j))
diseased_total = int(sum(cm[i].sum() for i in range(NUM_CLASSES)
                         if i not in healthy_idx
                         and CLASS_NAMES[i] != 'Background_without_leaves'))
print(f'Diseased leaves predicted healthy: {missed} of {diseased_total:,} '
      f'({missed / max(diseased_total, 1):.2%})')
print('The error that matters agronomically: a missed disease costs a crop, '
      'a false alarm costs a second look.')

## 18. Figure 8 — error budget by species

In [ ]:
species = [c.split('___')[0] if '___' in c else 'Background' for c in CLASS_NAMES]
uniq_species = sorted(set(species))
err_by_species = {s: 0 for s in uniq_species}
tot_by_species = {s: 0 for s in uniq_species}
for i in range(NUM_CLASSES):
    err_by_species[species[i]] += int(cm[i].sum() - cm[i, i])
    tot_by_species[species[i]] += int(cm[i].sum())

sp_order = sorted(uniq_species, key=lambda s: -err_by_species[s])
vals = [err_by_species[s] for s in sp_order]
total_err = max(sum(vals), 1)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(sp_order, vals, color=C_RED, alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + total_err * 0.02,
            f'{v}\n{v / total_err:.0%}', ha='center', fontsize=9,
            fontweight='bold')
ax.set_ylabel('Misclassified images')
ax.set_ylim(0, max(max(vals) * 1.25, 1))
ax.set_title('Fig 8 — Error budget by species', fontweight='bold')
ax.grid(axis='x', alpha=0)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig08_error_budget.png', bbox_inches='tight')
plt.show()

for s in sp_order:
    rate = err_by_species[s] / max(tot_by_species[s], 1)
    print(f'  {s:<14} {err_by_species[s]:>4} errors / {tot_by_species[s]:>5} '
          f'images  ({rate:.2%})')

## 19. Figure 9 — confidence calibration

The most directly useful figure for the app. PlantDoc shows a confidence
percentage to users, and nothing in v2 checked whether it means anything. A
model driven to 100% training accuracy is usually overconfident.

**ECE** is the average gap between stated confidence and observed accuracy.
Below ~0.05 is good. Label smoothing should improve this relative to v2.

In [ ]:
def calibration(conf, corr, n_bins=12):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    accs, confs, ns = [], [], []
    ece = 0.0
    for lo_, hi_ in zip(edges[:-1], edges[1:]):
        m = (conf > lo_) & (conf <= hi_)
        if not m.any():
            continue
        a, c, n = corr[m].mean(), conf[m].mean(), int(m.sum())
        accs.append(a); confs.append(c); ns.append(n)
        ece += n / len(conf) * abs(a - c)
    return np.array(accs), np.array(confs), np.array(ns), ece


accs, confs, ns, ece = calibration(y_conf, correct)

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.5))

ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.6, label='perfect calibration')
ax.plot(confs, accs, 'o-', color=C_BLUE, lw=2, ms=7, label='model')
for c_, a_ in zip(confs, accs):
    ax.plot([c_, c_], [c_, a_], color=C_RED, lw=1.5, alpha=0.5)
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.set_xlabel('Stated confidence')
ax.set_ylabel('Observed accuracy')
ax.set_title(f'(a) Reliability diagram — ECE = {ece:.4f}', fontweight='bold')
ax.legend(loc='upper left')

ax = axes[1]
bins = np.linspace(0, 1, 40)
ax.hist(y_conf[correct], bins=bins, color=C_GREEN, alpha=0.7,
        label=f'correct (n={int(correct.sum())})')
ax.hist(y_conf[~correct], bins=bins, color=C_RED, alpha=0.8,
        label=f'incorrect (n={int((~correct).sum())})')
ax.set_yscale('log')
ax.set_xlabel('Confidence')
ax.set_ylabel('Predictions (log scale)')
ax.set_title('(b) Confidence distribution', fontweight='bold')
ax.legend()

plt.suptitle('Fig 9 — Confidence calibration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig09_calibration.png', bbox_inches='tight')
plt.show()

print(f'ECE: {ece:.4f}   (below 0.05 is well calibrated)')
print(f'Mean confidence when correct   : {y_conf[correct].mean():.4f}')
print(f'Mean confidence when incorrect : {y_conf[~correct].mean():.4f}\n')
print('Threshold sweep — what the app gains by saying "not sure":')
print(f'{"threshold":>10}{"kept":>9}{"coverage":>11}{"acc@kept":>11}')
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    keep = y_conf >= t
    if keep.sum() == 0:
        continue
    print(f'{t:>10.2f}{int(keep.sum()):>9}{keep.mean():>10.1%}'
          f'{correct[keep].mean():>11.2%}')

## 20. Figure 10 — misclassified examples

The most confident mistakes, which are the most informative ones.

In [ ]:
wrong_idx = np.where(~correct)[0]
print(f'{len(wrong_idx)} misclassified images')

if len(wrong_idx):
    picks = wrong_idx[np.argsort(-y_conf[wrong_idx])][:16]
    rows = int(np.ceil(len(picks) / 4))
    fig, axes = plt.subplots(rows, 4, figsize=(15, 4.1 * rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, idx in zip(axes, picks):
        img = tf.io.decode_image(tf.io.read_file(test_p[idx]), channels=3,
                                 expand_animations=False)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method='bilinear')
        ax.imshow(tf.cast(img, tf.uint8).numpy())
        ax.set_title(f'true: {SHORT_NAMES[y_true[idx]]}\n'
                     f'pred: {SHORT_NAMES[y_pred[idx]]}  ({y_conf[idx]:.0%})',
                     fontsize=7.5, color=C_RED)
        ax.axis('off')
    for ax in axes[len(picks):]:
        ax.axis('off')

    plt.suptitle('Fig 10 — Most confident mistakes', fontsize=13,
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig10_misclassified.png', bbox_inches='tight')
    plt.show()

## 21. Figure 11 — convolutional feature maps

What the network sees at successive depths: the input, three convolutional
stages, and the pooled feature vector that feeds the classifier.

Early layers respond to edges and texture; deeper layers respond to
lesion-shaped structure. Each panel is the channel-mean activation, resized to a
common scale.

In [ ]:
bb = get_backbone(model)
conv_layers = [l for l in bb.layers
               if isinstance(l, (keras.layers.Conv2D,
                                 keras.layers.DepthwiseConv2D))]
picks = [conv_layers[int(f * (len(conv_layers) - 1))] for f in (0.05, 0.4, 0.9)]

pre_layers = model.layers[1:model.layers.index(bb)]
probe_idx = int(np.where(correct)[0][0]) if correct.any() else 0
raw = tf.io.decode_image(tf.io.read_file(test_p[probe_idx]), channels=3,
                         expand_animations=False)
raw = tf.clip_by_value(tf.image.resize(raw, (IMG_SIZE, IMG_SIZE),
                                       method='bilinear'), 0.0, 255.0)
x = tf.expand_dims(raw, 0)
for layer in pre_layers:
    x = layer(x, training=False)

feat_model = keras.Model(bb.inputs,
                         [l.output for l in picks] + [bb.output])
outs = feat_model(x, training=False)
maps, pooled = outs[:-1], outs[-1]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(tf.cast(raw, tf.uint8).numpy())
axes[0].set_title(f'(a) input\n{SHORT_NAMES[y_true[probe_idx]]}', fontsize=9)
axes[0].axis('off')

for k, (ax, fmap, layer) in enumerate(zip(axes[1:4], maps, picks), start=1):
    m = tf.reduce_mean(tf.cast(fmap[0], tf.float32), axis=-1).numpy()
    ax.imshow(m, cmap='viridis')
    ax.set_title(f'({chr(97 + k)}) conv stage {k}\n{layer.name}  '
                 f'{m.shape[0]}x{m.shape[1]}', fontsize=9)
    ax.axis('off')

vec = np.asarray(pooled[0], dtype=np.float32)
side = int(np.ceil(np.sqrt(vec.size)))
grid = np.pad(vec, (0, side * side - vec.size)).reshape(side, side)
axes[4].imshow(grid, cmap='inferno')
axes[4].set_title(f'(e) pooled features\n{vec.size}-d vector', fontsize=9)
axes[4].axis('off')

plt.tight_layout()
plt.suptitle('Fig 11 — Convolutional feature maps (channel mean)',
             fontsize=13, fontweight='bold', y=1.08)
plt.savefig(f'{WORK_DIR}/fig11_feature_maps.png', bbox_inches='tight')
plt.show()

## 22. Figure 12 — Grad-CAM

PlantVillage has a documented background-shortcut problem: images of one class
often share a background, so a model can score highly by reading the background
instead of the leaf. Grad-CAM shows where the evidence came from.

Heat on the lesions is the result you want. Heat on the background or the image
border means the accuracy number is not measuring what you think.

In [ ]:
def make_gradcam(model, img_batch):
    bb = get_backbone(model)
    last_conv = next(l for l in reversed(bb.layers) if len(l.output.shape) == 4)
    feat_model = keras.Model(bb.inputs, [last_conv.output, bb.output])

    bb_pos = model.layers.index(bb)
    x = tf.cast(img_batch, tf.float32)
    for layer in model.layers[1:bb_pos]:
        x = layer(x, training=False)

    with tf.GradientTape() as tape:
        conv_out, pooled = feat_model(x, training=False)
        tape.watch(conv_out)
        h = pooled
        for layer in model.layers[bb_pos + 1:]:
            h = layer(h, training=False)
        h = tf.cast(h, tf.float32)
        idx = tf.argmax(h[0])
        score = h[:, idx]

    grads = tape.gradient(score, conv_out)
    if grads is None:
        return None
    weights = tf.reduce_mean(tf.cast(grads, tf.float32), axis=(1, 2))
    cam = tf.reduce_sum(tf.cast(conv_out, tf.float32)
                        * weights[:, None, None, :], axis=-1)
    cam = tf.nn.relu(cam)[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], (IMG_SIZE, IMG_SIZE)).numpy()[..., 0]


try:
    rng = np.random.RandomState(SEED)
    picks = list(rng.choice(np.where(correct)[0],
                            size=min(4, int(correct.sum())), replace=False))
    if len(wrong_idx):
        picks += list(wrong_idx[np.argsort(-y_conf[wrong_idx])][:2])

    fig, axes = plt.subplots(2, len(picks), figsize=(3.1 * len(picks), 6.8))
    axes = np.atleast_2d(axes)

    for col, idx in enumerate(picks):
        raw = tf.io.decode_image(tf.io.read_file(test_p[idx]), channels=3,
                                 expand_animations=False)
        raw = tf.clip_by_value(tf.image.resize(raw, (IMG_SIZE, IMG_SIZE),
                                               method='bilinear'), 0.0, 255.0)
        cam = make_gradcam(model, tf.expand_dims(raw, 0))
        disp = tf.cast(raw, tf.uint8).numpy()

        ok = correct[idx]
        axes[0, col].imshow(disp)
        axes[0, col].set_title(SHORT_NAMES[y_true[idx]], fontsize=8,
                               color=C_GREEN if ok else C_RED)
        axes[0, col].axis('off')

        axes[1, col].imshow(disp)
        if cam is not None:
            axes[1, col].imshow(cam, cmap='jet', alpha=0.45)
        axes[1, col].set_title(f'pred: {SHORT_NAMES[y_pred[idx]]}\n'
                               f'{y_conf[idx]:.0%}', fontsize=8)
        axes[1, col].axis('off')

    plt.suptitle('Fig 12 — Grad-CAM (top: input, bottom: attribution; '
                 'green = correct, red = wrong)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig12_gradcam.png', bbox_inches='tight')
    plt.show()
    print('Check: heat should sit on lesions, not background or border.')
except Exception as exc:
    print(f'Grad-CAM skipped: {type(exc).__name__}: {exc}')
    print('Non-fatal. Everything else is unaffected.')

## 23. Classical ML baselines

Fits SVM, decision tree, logistic regression and k-NN on the same train/test
split so Fig 13 and Fig 14 are like-for-like.

Two honest caveats to carry into the write-up:

1. The baselines see **32x32 greyscale pixels**, not learned features. That is
   the standard comparison in the literature, and it is also why the gap looks
   large — the comparison measures "CNN vs raw pixels", not "CNN vs a
   well-engineered classical pipeline". Say so rather than implying the
   baselines were tuned.
2. Training is subsampled to `BASELINE_TRAIN_N` because SVM and k-NN scale
   badly. `LinearSVC` is used in place of an RBF kernel for the same reason.

In [ ]:
BASELINE_RESULTS = {}

if RUN_BASELINES:
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import LinearSVC
    from sklearn.tree import DecisionTreeClassifier

    def load_flat(paths, size=BASELINE_IMG_SIZE):
        def f(p):
            img = tf.io.decode_image(tf.io.read_file(p), channels=3,
                                     expand_animations=False)
            img = tf.image.resize(img, (size, size), method='bilinear')
            img = tf.image.rgb_to_grayscale(img)
            return tf.reshape(img, [-1]) / 255.0
        ds = (tf.data.Dataset.from_tensor_slices(paths)
              .map(f, num_parallel_calls=AUTOTUNE).batch(256))
        return np.concatenate([b.numpy() for b in ds], axis=0)

    t0 = time.time()
    rng = np.random.RandomState(SEED)
    sub = rng.choice(len(train_p), size=min(BASELINE_TRAIN_N, len(train_p)),
                     replace=False)
    Xtr, ytr = load_flat(train_p[sub]), train_l[sub]
    Xte, yte = load_flat(test_p), test_l
    print(f'Features: train {Xtr.shape}, test {Xte.shape} '
          f'({time.time() - t0:.0f}s)\n')

    BASELINE_MODELS = {
        'SVM': LinearSVC(C=0.01, max_iter=3000, dual='auto',
                         random_state=SEED),
        'Decision Tree': DecisionTreeClassifier(max_depth=24,
                                                random_state=SEED),
        'Logistic Regression': LogisticRegression(max_iter=400, n_jobs=-1,
                                                  random_state=SEED),
        'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    }

    for name, clf in BASELINE_MODELS.items():
        t0 = time.time()
        clf.fit(Xtr, ytr)
        pred = clf.predict(Xte)
        p, r, f, _ = precision_recall_fscore_support(
            yte, pred, average='macro', zero_division=0)
        BASELINE_RESULTS[name] = dict(
            accuracy=float((pred == yte).mean()), precision=float(p),
            recall=float(r), f1=float(f), pred=pred, clf=clf)
        print(f'  {name:<22} acc={BASELINE_RESULTS[name]["accuracy"]:.4f}  '
              f'F1={f:.4f}  ({time.time() - t0:.0f}s)')

    BASELINE_RESULTS['Proposed CNN'] = dict(
        accuracy=overall_acc, precision=float(precision.mean()),
        recall=float(recall.mean()), f1=float(f1.mean()),
        pred=y_pred, clf=None)
    print(f'  {"Proposed CNN":<22} acc={overall_acc:.4f}  '
          f'F1={f1.mean():.4f}')
else:
    print('RUN_BASELINES is False — skipping Fig 13 and Fig 14.')

## 24. Figure 13 — model comparison

Accuracy, macro precision, macro recall and macro F1 across all five models.

Each panel scales to its own data range rather than starting at zero, because
recall values cluster tightly and a 0-1 axis would render them identical — the
mistake that made several of v2's charts unreadable.

In [ ]:
if BASELINE_RESULTS:
    names = list(BASELINE_RESULTS.keys())
    metrics_4 = [('accuracy', 'Accuracy'), ('precision', 'Macro precision'),
                 ('recall', 'Macro recall'), ('f1', 'Macro F1')]
    colours = [C_BLUE, C_ORANGE, C_SAND, C_PURPLE, C_GREEN][:len(names)]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, (key, title) in zip(axes.flat, metrics_4):
        vals = [BASELINE_RESULTS[n][key] for n in names]
        bars = ax.bar(range(len(names)), vals, color=colours, alpha=0.9,
                      edgecolor='white')
        span = max(vals) - min(vals)
        pad = max(span * 0.35, 0.02)
        ax.set_ylim(max(0, min(vals) - pad), min(1.04, max(vals) + pad))
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + pad * 0.08,
                    f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
        ax.set_ylabel(title)
        ax.set_title(title, fontweight='bold')
        ax.grid(axis='x', alpha=0)

    plt.suptitle('Fig 13 — Proposed model vs classical ML baselines',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig13_model_comparison.png', bbox_inches='tight')
    plt.show()

    print(f'{"model":<22}{"accuracy":>10}{"precision":>11}{"recall":>9}{"F1":>9}')
    for n in names:
        r = BASELINE_RESULTS[n]
        print(f'{n:<22}{r["accuracy"]:>10.4f}{r["precision"]:>11.4f}'
              f'{r["recall"]:>9.4f}{r["f1"]:>9.4f}')

## 25. Figure 14 — ROC, CNN vs baselines

v2's ROC grid was useless because it compared one near-perfect model against
itself 20 times. Here ROC earns its place: the baselines span a wide AUC range,
so the curves separate.

Four representative classes, one per crop.

In [ ]:
if BASELINE_RESULTS and len(BASELINE_RESULTS) > 1:
    def scores_for(clf, X):
        if hasattr(clf, 'predict_proba'):
            return clf.predict_proba(X)
        return clf.decision_function(X)

    baseline_scores = {n: scores_for(r['clf'], Xte)
                       for n, r in BASELINE_RESULTS.items()
                       if r['clf'] is not None}

    y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
    wanted = ['Apple___Black_rot', 'Potato___Early_blight',
              'Strawberry___Leaf_scorch', 'Tomato___Tomato_mosaic_virus']
    show = [CLASS_NAMES.index(w) for w in wanted if w in CLASS_NAMES][:4]

    fig, axes = plt.subplots(2, 2, figsize=(13, 11))
    palette = {'Proposed CNN': C_BLUE, 'KNN': C_GREEN,
               'Decision Tree': C_RED, 'Logistic Regression': C_ORANGE,
               'SVM': C_PURPLE}

    for ax, ci in zip(axes.flat, show):
        series = [('Proposed CNN', y_prob[:, ci])]
        for n, sc in baseline_scores.items():
            series.append((n, sc[:, ci]))
        for n, sc in series:
            fpr, tpr, _ = roc_curve(y_bin[:, ci], sc)
            ax.plot(fpr, tpr, lw=2, color=palette.get(n, '#888'),
                    label=f'{n} (AUC = {auc(fpr, tpr):.3f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.7)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.set_xlabel('False positive rate')
        ax.set_ylabel('True positive rate')
        ax.set_title(SHORT_NAMES[ci], fontsize=10, fontweight='bold')
        ax.legend(loc='lower right', fontsize=8)

    plt.suptitle('Fig 14 — ROC curves, proposed model vs baselines',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig14_roc_comparison.png', bbox_inches='tight')
    plt.show()

    macro_auc = float(np.mean([
        auc(*roc_curve(y_bin[:, i], y_prob[:, i])[:2])
        for i in range(NUM_CLASSES)]))
    print(f'Proposed model macro AUC across all {NUM_CLASSES} classes: '
          f'{macro_auc:.4f}')
else:
    macro_auc = None

## 26. Ablation studies (Fig 15-18)

Off by default. Each configuration is a separate training run, so the whole
section costs roughly 60-90 minutes on a T4.

To keep it affordable these runs are **head-only, short, and on a subsample**
(`ABLATION_EPOCHS`, `ABLATION_TRAIN_N`). They measure *relative* effects, not
final accuracy — report them as trends and cite the main run for headline
numbers. Saying this plainly is stronger than implying every bar came from a
full run.

In [ ]:
ABLATIONS = {}

if RUN_ABLATIONS:
    rng = np.random.RandomState(SEED)
    sub = rng.choice(len(train_p), size=min(ABLATION_TRAIN_N, len(train_p)),
                     replace=False)
    ab_p, ab_l = train_p[sub], train_l[sub]
    n_runs = (len(ABLATION_DROPOUTS) + len(ABLATION_BATCH_SIZES) + 2
              + len(ABLATION_BACKBONES))
    print(f'{n_runs} runs x {ABLATION_EPOCHS} epochs on {len(ab_p):,} images.')

    def quick_train(tag, dropout=None, batch_size=None, augment=True,
                    backbone_name=None):
        keras.backend.clear_session()
        keras.utils.set_random_seed(SEED)
        rates = (dropout, dropout, dropout) if isinstance(dropout, float) \
            else (dropout or DROPOUT_RATES)
        m = build_model(NUM_CLASSES, trainable_backbone=False,
                        dropouts=rates, backbone_name=backbone_name)
        m.compile(optimizer=keras.optimizers.Adam(learning_rate=P1_LR),
                  loss=loss_fn,
                  metrics=[keras.metrics.CategoricalAccuracy(name='acc')])
        ds = make_train_ds(ab_p, ab_l, augment=augment, batch_size=batch_size)
        t0 = time.time()
        h = m.fit(ds, validation_data=val_ds, epochs=ABLATION_EPOCHS,
                  verbose=0)
        best = float(max(h.history['val_acc']))
        loss_ = float(min(h.history['val_loss']))
        print(f'  {tag:<34} val_acc={best:.4f}  val_loss={loss_:.4f}  '
              f'({time.time() - t0:.0f}s)')
        del m
        return best, loss_

    print('\nDropout sweep')
    ABLATIONS['dropout'] = {
        d: quick_train(f'dropout={d}', dropout=float(d))[0]
        for d in ABLATION_DROPOUTS}

    print('\nBatch size sweep')
    ABLATIONS['batch_size'] = {
        b: quick_train(f'batch_size={b}', batch_size=b)[0]
        for b in ABLATION_BATCH_SIZES}

    print('\nAugmentation')
    a_acc, a_loss = quick_train('augmented', augment=True)
    n_acc, n_loss = quick_train('non-augmented', augment=False)
    ABLATIONS['augmentation'] = {
        'Augmented': (a_acc, a_loss), 'Non-augmented': (n_acc, n_loss)}

    print('\nBackbone comparison')
    ABLATIONS['backbone'] = {}
    for name in ABLATION_BACKBONES:
        try:
            ABLATIONS['backbone'][name] = quick_train(
                f'backbone={name}', backbone_name=name)[0]
        except Exception as exc:
            print(f'  {name:<34} skipped: {type(exc).__name__}: {exc}')
else:
    print('RUN_ABLATIONS is False — skipping Fig 15-18.')
    print('Set RUN_ABLATIONS = True and re-run this section to produce them.')

In [ ]:
def ablation_bar(ax, labels, values, xlabel, title, colour, fmt='{:.4f}'):
    bars = ax.bar(range(len(labels)), values, color=colour, alpha=0.9,
                  edgecolor='white')
    span = max(values) - min(values)
    pad = max(span * 0.3, 0.02)
    ax.set_ylim(max(0, min(values) - pad), min(1.05, max(values) + pad))
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + pad * 0.07, fmt.format(v),
                ha='center', fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Validation accuracy')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0)


if ABLATIONS.get('dropout'):
    d = ABLATIONS['dropout']
    fig, ax = plt.subplots(figsize=(9, 5))
    ablation_bar(ax, [str(k) for k in d], list(d.values()),
                 'Dropout probability',
                 'Fig 15 — Validation accuracy vs dropout rate', C_SAND)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig15_dropout.png', bbox_inches='tight')
    plt.show()
    print(f'Best dropout: {max(d, key=d.get)}')

if ABLATIONS.get('batch_size'):
    d = ABLATIONS['batch_size']
    fig, ax = plt.subplots(figsize=(9, 5))
    ablation_bar(ax, [str(k) for k in d], list(d.values()), 'Batch size',
                 'Fig 16 — Validation accuracy vs batch size', C_ORANGE)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig16_batch_size.png', bbox_inches='tight')
    plt.show()
    print(f'Best batch size: {max(d, key=d.get)}')

if ABLATIONS.get('augmentation'):
    d = ABLATIONS['augmentation']
    labels = list(d)
    accs = [d[k][0] for k in labels]
    losses = [d[k][1] for k in labels]
    xpos = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(8.5, 5))
    ax.bar(xpos - 0.19, accs, 0.38, color=C_GREEN, label='Val accuracy')
    ax.bar(xpos + 0.19, losses, 0.38, color=C_BLUE, label='Val loss')
    for x_, v in zip(xpos - 0.19, accs):
        ax.text(x_, v + 0.015, f'{v:.4f}', ha='center', fontsize=9,
                fontweight='bold')
    for x_, v in zip(xpos + 0.19, losses):
        ax.text(x_, v + 0.015, f'{v:.4f}', ha='center', fontsize=9,
                fontweight='bold')
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, max(max(accs), max(losses)) * 1.25)
    ax.set_xlabel('Training data')
    ax.set_title('Fig 17 — Augmented vs non-augmented training',
                 fontweight='bold')
    ax.legend()
    ax.grid(axis='x', alpha=0)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig17_augmentation.png', bbox_inches='tight')
    plt.show()
    print(f'Augmentation delta: '
          f'{(d["Augmented"][0] - d["Non-augmented"][0]) * 100:+.2f} pts')

if ABLATIONS.get('backbone'):
    d = ABLATIONS['backbone']
    fig, ax = plt.subplots(figsize=(10, 5))
    ablation_bar(ax, list(d), list(d.values()), 'Backbone',
                 'Fig 18 — Validation accuracy by backbone '
                 f'(head-only, {ABLATION_EPOCHS} epochs)', C_TEAL)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/fig18_backbone.png', bbox_inches='tight')
    plt.show()
    print(f'Best backbone at this budget: {max(d, key=d.get)}')

## 27. Results summary

Replaces v2's five-bar chart, where every bar sat between 0.965 and 0.999 on a
0-1.12 axis and they were visually indistinguishable. Numbers belong in a
table.

In [ ]:
macro_p, macro_r = float(precision.mean()), float(recall.mean())
macro_f = float(f1.mean())
weighted_f = float(np.average(f1, weights=support))

rows = [
    ('Test accuracy', f'{overall_acc:.4f}'),
    ('Macro precision', f'{macro_p:.4f}'),
    ('Macro recall', f'{macro_r:.4f}'),
    ('Macro F1', f'{macro_f:.4f}'),
    ('Weighted F1', f'{weighted_f:.4f}'),
    ('Macro AUC', f'{macro_auc:.4f}' if macro_auc else 'n/a'),
    ('Expected calibration error', f'{ece:.4f}'),
    ('Test images', f'{len(y_true):,}'),
    ('Errors', f'{int((~correct).sum())}'),
]

print('=' * 52)
print(f'  FINAL PERFORMANCE — {MODEL_VERSION}')
print('=' * 52)
for k, v in rows:
    print(f'  {k:<32}{v:>16}')
print('=' * 52)
print('\nv2 baseline:')
print(f'  {"Test accuracy":<32}{"0.9651":>16}')
print(f'  {"Macro F1":<32}{"0.9680":>16}')
print(f'  {"Train/val gap":<32}{"0.0441":>16}')
print(f'\n  delta accuracy: {(overall_acc - 0.9651) * 100:+.2f} pts')

metrics = {
    'model_version': MODEL_VERSION,
    'backbone': BACKBONE,
    'test_accuracy': overall_acc,
    'macro_precision': macro_p,
    'macro_recall': macro_r,
    'macro_f1': macro_f,
    'weighted_f1': weighted_f,
    'macro_auc': macro_auc,
    'ece': float(ece),
    'tta_enabled': bool(use_tta_final),
    'accuracy_plain': acc_plain,
    'accuracy_tta': acc_tta,
    'n_test': int(len(y_true)),
    'n_errors': int((~correct).sum()),
    'label_smoothing': LABEL_SMOOTHING,
    'class_weight_mode': CLASS_WEIGHT_MODE,
    'dropout_rates': list(DROPOUT_RATES),
    'batch_size': BATCH_SIZE,
    'seed': SEED,
    'img_size': IMG_SIZE,
    'p1_epochs_run': int(p1_len),
    'p2_epochs_run': int(len(h2['acc'])),
    'lr_schedule': (f'ReduceLROnPlateau x{P2_LR_FACTOR:g}, '
                    f'patience {P1_LR_PATIENCE}/{P2_LR_PATIENCE}'),
    'lr_final': float(lr_hist[-1]) if lr_hist is not None else None,
    'lr_reductions': len(drops) if lr_hist is not None else None,
    'baselines': {k: {m: v[m] for m in
                      ('accuracy', 'precision', 'recall', 'f1')}
                  for k, v in BASELINE_RESULTS.items()},
    'ablations': {k: (v if not isinstance(v, dict) else
                      {str(kk): vv for kk, vv in v.items()})
                  for k, v in ABLATIONS.items()},
    'per_class': {
        CLASS_NAMES[i]: {'precision': float(precision[i]),
                         'recall': float(recall[i]),
                         'f1': float(f1[i]),
                         'support': int(support[i])}
        for i in range(NUM_CLASSES)},
}
with open(f'{WORK_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=float)
print('\nWrote metrics.json')

## 28. Export

Writes the model, class list and metrics, then prints the backend changes this
model requires.

In [ ]:
with open(f'{WORK_DIR}/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

artifacts = (sorted(Path(WORK_DIR).glob('fig*.png'))
             + sorted(Path(WORK_DIR).glob('tbl*.png'))
             + [Path(BEST_MODEL_PATH),
                Path(f'{WORK_DIR}/class_names.json'),
                Path(f'{WORK_DIR}/metrics.json'),
                Path(f'{WORK_DIR}/p1_log.csv'),
                Path(f'{WORK_DIR}/p2_log.csv')])
print('Artifacts:')
for p in artifacts:
    mark = 'ok ' if p.exists() else 'MISSING'
    size = p.stat().st_size / 1024 if p.exists() else 0
    print(f'  [{mark}] {p.name:<38} {size:>10,.1f} KB')

print('\n' + '=' * 72)
print('BACKEND CHANGE REQUIRED BEFORE DEPLOYING THIS MODEL')
print('=' * 72)
print("""
This model has preprocessing BAKED IN. It expects raw RGB in [0, 255].
The current backend applies mobilenet_v2.preprocess_input first. If you deploy
without the change below, preprocessing runs twice and every prediction is
wrong -- confidently wrong, with no error raised.

In backend/app/services/disease_detection.py:

  - from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
  ...
    def preprocess(self, image_rgb):
        size = self.settings.model_input_size
  -     resized = cv2.resize(image_rgb, (size, size), interpolation=cv2.INTER_AREA)
  -     batch = np.expand_dims(resized.astype(np.float32), axis=0)
  -     return preprocess_input(batch)
  +     # v3 bakes preprocessing into the model: feed raw RGB 0-255.
  +     # INTER_LINEAR matches the bilinear resize used in training.
  +     resized = cv2.resize(image_rgb, (size, size), interpolation=cv2.INTER_LINEAR)
  +     return np.expand_dims(resized.astype(np.float32), axis=0)

In backend/app/core/config.py, bump:

      model_version: str = "REPLACE_ME"

Then: copy plant_best_model.keras to model/, class_names.json to Resources/,
and run the backend tests.
""".replace('REPLACE_ME', MODEL_VERSION))
print('=' * 72)